In [ ]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd
# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

df = pd.read_csv('npr.csv')
documents = df['Article'].tolist()

stop_words = set(stopwords.words('english')) # Create a set of English stopwords
lemmatizer = WordNetLemmatizer() # Initialize a WordNet lemmatizer
def preprocess_text(text):
 tokens = word_tokenize(text.lower()) # Tokenize the text into words and convert to lowercase
 tokens = [token for token in tokens if token.isalnum()] # Filter out non-alphanumeric tokens
 tokens = [token for token in tokens if token not in stop_words] # Remove stopwords from the tokens
 tokens = [lemmatizer.lemmatize(token) for token in tokens] # Lemmatize each token
 return tokens # Return the preprocessed tokens
preprocessed_documents = [preprocess_text(doc) for doc in documents] # Preprocess each document in the list
print(preprocessed_documents[0])

# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)
# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)
# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents] 

# Run LDA
lda_model = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15) # Train an LDA modelon the corpus with 2 topics using Gensim's LdaModel class

# empty list to store dominant topic labels for each document
article_labels = []
# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
 # for each document, convert to bag-of-words representation
 bow = dictionary.doc2bow(doc)
 # get list of topic probabilities
 topics = lda_model.get_document_topics(bow)
 # determine topic with highest probability
 dominant_topic = max(topics, key=lambda x: x[1])[0]
 # append to the list
 article_labels.append(dominant_topic)
# Create DataFrame
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})
# Print the DataFrame
print("Table with Articles and Topic:")
print(df_result)
print()



[nltk_data] Downloading package stopwords to
[nltk_data]     /home/d3f6f63e-2634-4630-b3e8-
[nltk_data]     bd2f955ed26a/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/d3f6f63e-2634-4630-b3e8-
[nltk_data]     bd2f955ed26a/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/d3f6f63e-2634-4630-b3e8-
[nltk_data]     bd2f955ed26a/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


['washington', '2016', 'even', 'policy', 'bipartisan', 'politics', 'sense', 'year', 'show', 'little', 'sign', 'ending', 'president', 'obama', 'moved', 'sanction', 'russia', 'alleged', 'interference', 'election', 'concluded', 'republican', 'long', 'called', 'similar', 'severe', 'measure', 'could', 'scarcely', 'bring', 'approve', 'house', 'speaker', 'paul', 'ryan', 'called', 'obama', 'measure', 'appropriate', 'also', 'overdue', 'prime', 'example', 'administration', 'ineffective', 'foreign', 'policy', 'left', 'america', 'weaker', 'eye', 'gop', 'leader', 'sounded', 'much', 'theme', 'urging', 'president', 'obama', 'year', 'take', 'strong', 'action', 'deter', 'russia', 'worldwide', 'aggression', 'including', 'operation', 'wrote', 'devin', 'nunes', 'chairman', 'house', 'intelligence', 'committee', 'week', 'left', 'office', 'president', 'suddenly', 'decided', 'stronger', 'measure', 'indeed', 'appearing', 'cnn', 'frequent', 'obama', 'critic', 'trent', 'frank', 'called', 'much', 'tougher', 'acti

In [4]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')  # FIX: needed for lemmatizer

# Load data
df = pd.read_csv('npr.csv')

# FIX: ensure no null + correct type
df = df[['Article']].dropna()
df['Article'] = df['Article'].astype(str)

documents = df['Article'].tolist()

# Preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [token for token in tokens if token.isalnum()]
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]

print(preprocessed_documents[0])

# Create Dictionary & Corpus
dictionary = corpora.Dictionary(preprocessed_documents)

# FIX: reduce threshold to avoid empty dictionary
dictionary.filter_extremes(no_below=10, no_above=0.5)

corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

# Run LDA
lda_model = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)

# Assign dominant topics
article_labels = []

for doc in preprocessed_documents:
    bow = dictionary.doc2bow(doc)
    topics = lda_model.get_document_topics(bow)
    
    if topics:
        dominant_topic = max(topics, key=lambda x: x[1])[0]
    else:
        dominant_topic = -1  # FIX: avoid crash
    
    article_labels.append(dominant_topic)

# Create DataFrame
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})

print("Table with Articles and Topic:")
print(df_result)
print()

# Print top terms for each topic
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    print([term[0] for term in top_terms])
    print()

# FIXED: correct indentation + safe split
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    
    terms = [term.strip() for term in topic.split("+")]
    
    for term in terms:
        if "*" in term:  # FIX: avoid split error
            weight, word = term.split("*")
            print(f"- {word.strip()} (weight: {weight.strip()})")
    
    print()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/d3f6f63e-2634-4630-b3e8-
[nltk_data]     bd2f955ed26a/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/d3f6f63e-2634-4630-b3e8-
[nltk_data]     bd2f955ed26a/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/d3f6f63e-2634-4630-b3e8-
[nltk_data]     bd2f955ed26a/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/d3f6f63e-2634-4630-b3e8-
[nltk_data]     bd2f955ed26a/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


ParserError: Error tokenizing data. C error: EOF inside string starting at row 227

In [ ]:
# Print top terms for each topic
for topic_id in range(lda_model.num_topics):
    print(f"Top terms for Topic #{topic_id}:")
    
    top_terms = lda_model.show_topic(topic_id, topn=10)
    
    print([term[0] for term in top_terms])
    print()

In [3]:
# Print the top terms for each topic with weight
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
 print(f"Topic {idx}:")
 terms = [term.strip() for term in topic.split("+")]
 for term in terms:
 weight, word = term.split("*")
 print(f"- {word.strip()} (weight: {weight.strip()})")
 print()

IndentationError: expected an indented block after 'for' statement on line 6 (932834530.py, line 7)